# Deploying and Evaluating the Agent

Notebook 01 left you with a working agent and no way to give it to anyone. It
lives in a notebook, it holds an open Neo4j driver, and it reads your
credentials from `dbutils`. None of that survives being handed to a colleague.

Here you take the same graph, log it as an MLflow model, deploy it to a Model
Serving endpoint, and score it against a question set with MLflow's judges.

**Prerequisites**

| Lab | What this notebook needs from it |
|---|---|
| [Notebook 01](01_langgraph_agent.ipynb) | A run that reached the end. Everything here uses the same three tools |
| [Lab 1 notebook 02](../Lab_1_Aura_Setup/02_credentials_and_cypher.ipynb) | The `fleet-ops-<your-user>` secret scope |
| [Lab 4 Part A](../Lab_4_Compound_AI_Agents/04_genie_agent.ipynb) | Your Genie Agent, and its agent ID |

**Learning objectives**

- See the three things a notebook agent assumes that a served agent cannot
- List the Databricks objects the endpoint may read, and pass the Aura credentials a different way
- Log a models-from-code agent to Unity Catalog and deploy it
- Query the endpoint and confirm all three tools still answer
- Score routing with a deterministic scorer and answers with an LLM judge


## What changes when the agent leaves the notebook

Notebook 01 makes three assumptions that only hold inside a notebook.

**`dbutils` exists.** Notebook 01 reads your Aura password out of the secret
scope with `dbutils.secrets.get`. A serving container has no `dbutils` and no
notebook user. It gets its credentials as environment variables instead. You set
those variables to secret references at deploy time, and Model Serving resolves
them to real values when the endpoint starts.

**You are the one asking.** In the notebook, every Databricks call runs as you,
with your permissions. The endpoint runs as a service principal that Model
Serving creates, and that principal starts with no permissions at all. You say
what it may read by listing those objects when you log the model, and the deploy
grants them from that list. Section 3 is where you build the list.

**The graph is already built.** In the notebook you ran the cells that opened
the connections and wired the nodes. The endpoint has no cells. It has to build
the whole graph itself, on startup, on a machine you never see. So the wiring
moves out of the notebook and into `agent.py`, and MLflow loads that file as the
model.

`agent.py` sits beside `tools.py` in this folder. Open it. It imports the same
node builders notebook 01 used and wires the same graph. Two things are new:
`build_runtime`, which opens the connections from environment variables, and
`FleetOpsAgent`, which is the MLflow interface Model Serving speaks.


## Section 1: Configuration

The same Genie Agent ID as notebook 01, and the same secret scope, derived from
`current_user()` so there is nothing to copy across.

The warehouse ID comes out of that scope. Notebook 01 read it off your Genie
Agent and stored it there, so this notebook reads it back rather than asking you
to find it. Section 3 has to name that warehouse, and it has to name the one the
agent actually runs on.

The model name and the endpoint name are built from that scope by `agent.py`
rather than typed here. Both carry your slug, so a workshop workspace holds one
model and one endpoint per participant and nobody overwrites anybody. Lab 6
redeploys **this** endpoint with memory added and registers into **this** model
rather than standing up a second pair, so both names are a contract between the
two labs.

In [ ]:
# ==================================================
# CONFIGURATION - replace GENIE_AGENT_ID with yours
# ==================================================

# From Lab 4 Part A. Open your Genie Agent and take the ID out of the URL:
#   https://<workspace>/genie/rooms/<GENIE_AGENT_ID>
GENIE_AGENT_ID = ""

import sys

sys.path.insert(0, ".")

from agent import endpoint_name, model_name
from tools import read_warehouse_id, secret_scope_name

CURRENT_USER = spark.sql("SELECT current_user()").collect()[0][0]
SECRET_SCOPE = secret_scope_name(spark)
UC_MODEL_NAME = model_name(SECRET_SCOPE)
ENDPOINT_NAME = endpoint_name(SECRET_SCOPE)

# The warehouse Genie runs its SQL on. Notebook 01 read it off your Genie Agent
# and stored it in this scope, so it is read back rather than typed: the
# resource declared in Section 3 has to be the warehouse the agent actually
# uses. Secret values are redacted in notebook output, so this one prints as
# [REDACTED] and is intact in Python.
WAREHOUSE_ID = ""
try:
    WAREHOUSE_ID = read_warehouse_id(dbutils, SECRET_SCOPE)
except RuntimeError as error:
    print(error)

print(f"Secret scope:   {SECRET_SCOPE}")
print(f"Genie Agent ID: {GENIE_AGENT_ID}")
print(f"Warehouse ID:   {WAREHOUSE_ID or '(missing)'}")
print(f"UC model:       {UC_MODEL_NAME}")
print(f"Endpoint:       {ENDPOINT_NAME}")
if not GENIE_AGENT_ID:
    print("\nGENIE_AGENT_ID is empty. Section 2 onward needs it.")

## Section 2: Run the agent before you log it

A deploy takes about fifteen minutes and tells you very little when it fails.
Running `agent.py` here first costs a minute and fails with a stack trace, so do
that.

`export_neo4j_env` reads the values out of your secret scope and writes them
into this process's environment, under the same variable names the endpoint will
use. The password is read, written, and dropped inside that call, so no cell
here holds it in a variable.

`build_runtime` then does exactly what the serving container does at startup:
open the driver, read the database name out of the environment, build the three
nodes, drop `graphrag_node` if the vector index is missing, and compile the
graph.


In [ ]:
from agent import DEFAULT_CONFIG, build_runtime, export_neo4j_env

print("Environment variables set:", export_neo4j_env(dbutils, SECRET_SCOPE))

MODEL_CONFIG = dict(DEFAULT_CONFIG)
MODEL_CONFIG["genie_agent_id"] = GENIE_AGENT_ID

runtime = build_runtime(MODEL_CONFIG)
print("Tools available to the supervisor:", runtime.available_tools)

state = runtime.graph.invoke(
    {
        "question": "What is the procedure for an EGT exceedance?",
        "trace": [],
        "findings": [],
    }
)
print("Route:", state["trace"])
print(state["answer"][:600])

runtime.driver.close()


## Section 3: List the Databricks objects the agent uses

The endpoint runs as a service principal with no permissions at all. It can read
a Databricks object only if you name that object when you log the model. MLflow
saves the list with the model, and `agents.deploy` then grants the principal
access to every item on it. MLflow calls these objects **resources**, which is
where the `build_resources` and `resources=` names below come from.

The agent uses twelve:

| Count | Object | Why the agent needs it |
|---|---|---|
| 1 | Your Genie Agent | `genie_node` sends it the question |
| 1 | The SQL warehouse behind that Genie Agent | Runs the SQL that Genie writes |
| 8 | The gold tables in `databricks-neo4j-workshop.aircraft` | That SQL reads them |
| 1 | `databricks-claude-sonnet-5` | The supervisor and the two nodes that write text call it |
| 1 | `databricks-bge-large-en` | Turns the question into the vector `graphrag_node` searches with |

`build_resources` in `agent.py` builds the list, so the cell below is one call.
Lab 6 redeploys this same endpoint and calls the same function, which keeps both
deploys on the same twelve objects.

**Name the warehouse, not just the Genie Agent.** Naming the Genie Agent grants
access to the Genie Agent only. The warehouse underneath it is a separate object
and needs its own grant. Leave it out and the endpoint deploys, starts, and
routes correctly, and then every sensor question comes back with `is not
authorized to use or monitor this SQL Endpoint`. Naming the wrong warehouse ID
gives you the same error. That is why notebook 01 stores the correct ID in your
secret scope and Section 1 reads it back instead of asking you to type it.

**Aura is not on this list.** Only Databricks objects can be named and granted
this way, and your Aura instance is not one. It reaches the endpoint a different
way: as a URI, username, password, and database name, passed in Section 5 as
secret references. Databricks objects get named here and granted for you.
Everything else is a credential you have to hand over yourself.

In [ ]:
from agent import build_resources

# Both IDs come from Section 1, and both have to be set: an empty one is
# accepted here and then fails the deploy.
if not GENIE_AGENT_ID or not WAREHOUSE_ID:
    raise ValueError(
        "GENIE_AGENT_ID and WAREHOUSE_ID must both be set. Rerun Section 1 and "
        "read what it printed."
    )

RESOURCES = build_resources(GENIE_AGENT_ID, WAREHOUSE_ID)
for resource in RESOURCES:
    print(type(resource).__name__, resource.to_dict())

## Section 4: Log the model

`python_model="agent.py"` is the models-from-code pattern. MLflow stores the
file rather than a pickle of an object, so what gets deployed is source you can
read, and nothing has to survive being serialized. `agent.py` ends with
`set_model(AGENT)`, which is the line that makes it the model.

`code_paths` carries `tools.py` and Lab 3's `data_utils.py` into the artifact.
The agent needs both. `tools.py` is where the nodes and the supervisor prompt
live, and `data_utils.py` is where the embedder comes from, the same one that
wrote the vectors in your index.

**Pin `pip_requirements` rather than letting MLflow infer them.** Inference
reads the environment this notebook is running in, and your cluster carries
libraries the agent never imports. One of them, the Lab 6 memory wheel, has a
version with a local segment, `0.5.1.dev1+mentions`. No package index can
resolve a local segment, so an inferred requirement naming it produces a
container that cannot be built, and you find out fifteen minutes later in a
build log. The list below is what the agent actually imports, at the versions
this workshop installs.


In [ ]:
import mlflow

PIP_REQUIREMENTS = [
    "mlflow>=3.1.0",
    "neo4j==6.2.0",
    "neo4j-graphrag>=1.17.0",
    "langgraph==1.2.4",
    "langchain-core>=1.4.6",
    "pydantic==2.13.4",
    "databricks-sdk>=0.60.0",
]

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="fleet-ops-assistant"):
    logged = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        code_paths=["tools.py", "../Lab_3_Semantic_Search/data_utils.py"],
        model_config=MODEL_CONFIG,
        resources=RESOURCES,
        pip_requirements=PIP_REQUIREMENTS,
        registered_model_name=UC_MODEL_NAME,
    )

MODEL_VERSION = logged.registered_model_version
print(f"Registered {UC_MODEL_NAME} version {MODEL_VERSION}")


Read back what was logged. `requirements.txt` in the artifact is the file the
serving container installs from, and it is the last cheap place to catch a
requirement that cannot resolve.


In [ ]:
import pathlib

artifact = mlflow.artifacts.download_artifacts(artifact_uri=logged.model_uri)
print((pathlib.Path(artifact) / "requirements.txt").read_text())
print("Files in the model:")
for path in sorted(pathlib.Path(artifact).rglob("*")):
    if path.is_file():
        print(" ", path.relative_to(artifact))


## Section 5: Pass the Aura credentials as secret references

Section 3 named the Databricks objects and MLflow granted them. Aura gets no
such treatment: there is nothing to name and nothing to grant, because Aura is
not a Databricks object. The endpoint needs the actual URI, username, password,
and database name.

They travel as **secret references**. A secret reference is the string
`{{secrets/<scope>/<key>}}`. You put that string in the endpoint's environment
block, and Model Serving swaps it for the real value when it applies the
configuration. So the endpoint's saved configuration holds the reference, and
only the running container holds the value. Your password never appears in this
notebook, in MLflow, or in the endpoint config.

`serving_environment_vars` builds all four references from your scope name. The
cell below prints them. They are safe to print because they are references, not
the secrets themselves.


In [ ]:
from agent import serving_environment_vars

ENVIRONMENT_VARS = serving_environment_vars(SECRET_SCOPE)
for name, reference in ENVIRONMENT_VARS.items():
    print(f"{name} = {reference}")


### Deploy

`agents.deploy` creates the endpoint, attaches the model version, applies the
environment block, and grants the serving principal the objects you listed in
Section 3.

Set an active MLflow experiment first. That is what makes the deployed agent
send its traces back to MLflow. Skip it and the endpoint still answers, but you
get no traces, and the trace is where you see which tool ran.

The call returns in under a minute. The endpoint is not ready yet when it does.


In [ ]:
from databricks import agents

mlflow.set_experiment(f"/Users/{CURRENT_USER}/{ENDPOINT_NAME}")

deployment = agents.deploy(
    UC_MODEL_NAME,
    MODEL_VERSION,
    endpoint_name=ENDPOINT_NAME,
    environment_vars=ENVIRONMENT_VARS,
    scale_to_zero=True,
)
print("Endpoint:", deployment.endpoint_name)
print("Query URL:", deployment.query_endpoint)


## Section 6: Wait for it

A first deploy builds a container from the requirements you pinned, which takes
roughly ten to fifteen minutes. The cell below polls until the endpoint is ready
and prints where it got to, so you can watch instead of guessing.


In [ ]:
import time

from databricks.sdk import WorkspaceClient

workspace = WorkspaceClient()
started = time.time()

while True:
    endpoint = workspace.serving_endpoints.get(ENDPOINT_NAME)
    ready = endpoint.state.ready.value
    updating = endpoint.state.config_update.value
    print(f"[{time.time() - started:6.0f}s] ready={ready} config_update={updating}")
    if updating == "UPDATE_FAILED":
        raise RuntimeError("Deployment failed. Open the endpoint page and read the build logs.")
    if ready == "READY" and updating == "NOT_UPDATING":
        break
    if time.time() - started > 1800:
        raise TimeoutError("Endpoint did not become ready within 30 minutes.")
    time.sleep(30)

print(f"\nReady after {(time.time() - started) / 60:.1f} minutes.")


## Section 7: Ask the endpoint

The endpoint speaks the Responses API, so a question goes in as one user message
and the answer comes back as output items. `custom_outputs` carries the route,
which is the list of tools the supervisor called.

Read the route first, then the answer. A bad answer with a sensible route is a
tool or prompt problem. A bad answer with a strange route is a routing problem,
and those live in different files.


In [ ]:
from mlflow.deployments import get_deploy_client

deploy_client = get_deploy_client("databricks")


def ask_endpoint(question: str) -> dict:
    """Send one question to the endpoint and unpack the reply."""
    response = deploy_client.predict(
        endpoint=ENDPOINT_NAME,
        inputs={"input": [{"role": "user", "content": question}]},
    )
    text = "".join(
        part.get("text", "")
        for item in response.get("output", [])
        for part in item.get("content", [])
    )
    custom = response.get("custom_outputs") or {}
    return {
        "answer": text,
        "trace": custom.get("trace", []),
        "usage": response.get("usage") or {},
    }


result = ask_endpoint("What is the procedure for an EGT exceedance?")
print("Route:", result["trace"])
print(result["answer"])


One question per tool, through the endpoint rather than through the graph in
this notebook. Same three tools, same routing prompt, none of your credentials.

The check is whether the tool that could answer was called, not whether it was
the only one. A tool that comes back empty sends the supervisor round again, so
a route of two or three names on a single-tool question is the retry working
rather than the routing failing.


In [ ]:
PER_TOOL_QUESTIONS = [
    ("genie_node", "What is the average EGT for aircraft N10000?"),
    ("cypher_node", "Which components are in the hydraulic system of N10000?"),
    ("graphrag_node", "What is the procedure for an EGT exceedance?"),
]

for expected, question in PER_TOOL_QUESTIONS:
    result = ask_endpoint(question)
    called = result["trace"]
    verdict = "OK " if expected in called else "HUH"
    print(f"{verdict} {expected:14} -> {called}")
    print(f"     {question}")


### The anchor question

The one that needs all three. Sensor readings live only in Delta, maintenance
history lives only in the graph, and the procedure lives only in the manuals, so
no single tool can finish it.


In [ ]:
ANCHOR = (
    "Which engines are showing abnormal EGT readings, what maintenance history "
    "do those aircraft have, and what does the maintenance manual say to do "
    "about high EGT?"
)

anchor_result = ask_endpoint(ANCHOR)
print("Route:", anchor_result["trace"])
print()
print(anchor_result["answer"])


## Section 8: Evaluate it

Score two things, and score them separately, because they fail for different
reasons.

**Routing** is a yes or no. Either the supervisor called a tool that could
answer the question, or it did not, and the trace already says which. That needs
a plain Python function, decorated as a scorer, not a judge.

**The answer** has no single right form, so it gets an LLM judge. `Correctness`
checks the answer against the facts you said a good answer contains.
`RelevanceToQuery` catches the answer that is true but answers a different
question.

Both run over the same evaluation set. `predict_fn` calls the endpoint, so the
scores describe the deployed model, not a copy of it running in this notebook.


In [ ]:
EVAL_PAIRS = [
    {
        "inputs": {"question": "What is the average EGT for aircraft N10000?"},
        "expectations": {
            "expected_tools": ["genie_node"],
            "expected_facts": ["an average EGT value in degrees Celsius for N10000"],
        },
    },
    {
        "inputs": {
            "question": "Which components are in the hydraulic system of N10000?"
        },
        "expectations": {
            "expected_tools": ["cypher_node"],
            "expected_facts": ["a pump", "a filter", "a reservoir", "an actuator"],
        },
    },
    {
        "inputs": {"question": "What is the procedure for an EGT exceedance?"},
        "expectations": {
            "expected_tools": ["graphrag_node"],
            "expected_facts": [
                "borescope inspection of the high pressure turbine",
                "verifying EGT probe calibration",
            ],
        },
    },
    {
        "inputs": {
            "question": "What maintenance events has aircraft N10004 had, and how severe were they?"
        },
        "expectations": {
            "expected_tools": ["cypher_node"],
            "expected_facts": ["maintenance events for N10004 with a severity"],
        },
    },
    {
        "inputs": {"question": "How do I troubleshoot engine vibration?"},
        "expectations": {
            "expected_tools": ["graphrag_node"],
            "expected_facts": ["a vibration troubleshooting step from the manual"],
        },
    },
    {
        "inputs": {"question": ANCHOR},
        "expectations": {
            "expected_tools": ["genie_node", "cypher_node", "graphrag_node"],
            "expected_facts": [
                "engines with elevated EGT",
                "maintenance history for those aircraft",
                "what the manual says to do about high EGT",
            ],
        },
    },
]
print(f"{len(EVAL_PAIRS)} evaluation pairs")


In [ ]:
from mlflow.genai.scorers import Correctness, RelevanceToQuery, scorer


@scorer
def routing(outputs: dict, expectations: dict) -> float:
    """Fraction of the expected tools the supervisor actually called.

    Deterministic on purpose. A judge asked whether the right tool ran would be
    guessing at something the trace already states.
    """
    expected = set(expectations.get("expected_tools", []))
    if not expected:
        return 1.0
    called = set(outputs.get("trace", []))
    return len(expected & called) / len(expected)


def predict_fn(question: str) -> dict:
    """Send one evaluation question to the deployed endpoint.

    Returns the answer and the route together, because the two scorers need
    different halves of it and MLflow hands both the same `outputs`.
    """
    result = ask_endpoint(question)
    return {"answer": result["answer"], "trace": result["trace"]}


results = mlflow.genai.evaluate(
    data=EVAL_PAIRS,
    predict_fn=predict_fn,
    scorers=[routing, Correctness(), RelevanceToQuery()],
)
results.tables["eval_results"]


Open the run in the MLflow experiment to read the traces beside the scores. A
low `Correctness` with `routing` at 1.0 means the right tool ran and answered
badly, so the fix is in the prompt or the data. A low `routing` means the
supervisor never called the tool that had the answer, so the fix is in the
supervisor prompt, which is Section 6 of notebook 01.


## Section 9: When it fails

Five failures account for nearly all of them, and each has one thing to check.

**A permission denied on the `agents` schema, in Section 4 or in the deploy.**
Nothing you did. The schema exists and your class was never granted one of the
two privileges Lab 5 needs in it. Section 4 fails with `PERMISSION_DENIED: User
does not have CREATE MODEL on Schema 'databricks-neo4j-workshop.agents'`, and
the deploy fails later with `Insufficient permission to create tables in
'databricks-neo4j-workshop.agents'`, because `agents.deploy` creates inference
tables beside the model it serves and there is no way to ask it not to. An
administrator fixes both for everybody with two statements, and you can rerun
from Section 4 straight after:

```sql
GRANT CREATE MODEL ON SCHEMA `databricks-neo4j-workshop`.`agents` TO `account users`;
GRANT CREATE TABLE ON SCHEMA `databricks-neo4j-workshop`.`agents` TO `account users`;
```

**The endpoint answers, but says it could not open its Neo4j connection.** The
model loaded and the credentials did not arrive. Open the endpoint page, look
under the served entity, and check that all four environment variables are there
and that the scope they name is the `fleet-ops-<your-user>` scope you created in
Lab 1 notebook 02. The agent prints the missing variable by name, so read the
message before changing anything.

**Section 1 says the warehouse ID is missing, or the deploy fails with `MLModel
file contains an invalid dependency name (null or empty string) in dependency
type 'sql_warehouse'`.** Both are the same thing: nothing is stored under
`sql-warehouse-id` in your secret scope. Rerun Section 1 of notebook 01, which
reads the warehouse off your Genie Agent and stores it, then rerun Section 1
here.

**`genie_node` says it is not authorized to use or monitor this SQL Endpoint.**
The `DatabricksSQLWarehouse` resource names the wrong warehouse, which means the
stored ID is not the one your Genie Agent runs on. Set `WAREHOUSE_ID` by hand in
Section 1 of notebook 01, rerun that cell to store it, then log and deploy
again. Nothing else has to change, and the routing you already verified will be
unaffected.

**The deploy never reaches READY.** The container could not be built, and the
reason is in the build logs on the endpoint page rather than in this notebook. A
requirement that cannot resolve is the usual cause, which is why Section 4 pins
them and prints the file back.

An endpoint deployed with `scale_to_zero=True` sleeps when it is idle, so the
first question after a quiet period takes longer while it wakes. That is not a
failure.

## What you built

The same agent from notebook 01, now running as a service anyone can call.

What changed is whose permissions it runs on. In the notebook, every query the
agent made was authorized as you, so your Unity Catalog grants decided what it
could read. The endpoint runs on a service principal instead, which starts with
access to nothing and ends up with access to exactly the objects you listed in
Section 3. Anything else in the workspace is unreachable from it, including
tables you can read yourself.

Three ideas are worth taking away:

- **Databricks objects and outside credentials are handled differently.** You
  list the objects and the platform grants them. You pass the credentials
  yourself, as secret references.
- **A served model is source code, not a pickle.** `agent.py` is what gets
  deployed, so you can read exactly what runs.
- **Score routing apart from answers.** They fail for different reasons, and the
  fix for each lives in a different file.

Lab 6 redeploys this same endpoint with memory in Neo4j, so it can be asked a
follow-up.
